# ScanOps V12-7B — Colab에서 GGUF 변환 (로컬 디스크 우회)

로컬 맥은 디스크가 부족해 7B 변환이 안 됨. **Colab에서 merge→GGUF→Q4까지** 만들고
최종 **Q4 GGUF(~4.5GB) 하나만 다운로드**한다. 로컬에선 `ollama create`만 하면 끝.

런타임: **T4 GPU** (merge에 GPU 사용). 소요 ~10~15분.


## ① 의존성 + llama.cpp 빌드 (양자화 바이너리용)


In [ ]:
!pip -q install transformers peft accelerate sentencepiece
!git clone -q https://github.com/ggerganov/llama.cpp
!pip -q install -r llama.cpp/requirements.txt
# 양자화 바이너리(llama-quantize) 빌드 (~4분)
!cd llama.cpp && cmake -B build -DGGML_NATIVE=OFF > /dev/null 2>&1 && cmake --build build --config Release -j --target llama-quantize > /dev/null 2>&1
!ls llama.cpp/build/bin/llama-quantize && echo OK

## ② adapter_v12_7b.zip 업로드 + 압축 해제


In [ ]:
import os, shutil
if os.path.exists('adapter7b'): shutil.rmtree('adapter7b')
os.makedirs('adapter7b', exist_ok=True)
from google.colab import files
up = files.upload()   # adapter_v12_7b.zip 선택
import zipfile
zname = [k for k in up if k.endswith('.zip')][0]
with zipfile.ZipFile(zname) as z: z.extractall('adapter7b')
print('adapter ok:', os.path.exists('adapter7b/adapter_model.safetensors'))

## ③ 베이스 7B + 어댑터 병합 → merged 저장 (GPU)


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
BASE='Qwen/Qwen2.5-Coder-7B-Instruct'
tok=AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
base=AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map='cpu', trust_remote_code=True)
merged=PeftModel.from_pretrained(base, 'adapter7b').merge_and_unload()
merged.save_pretrained('merged7b', safe_serialization=True)
tok.save_pretrained('merged7b')
print('merged 저장 완료')

## ④ GGUF f16 변환 + Q4_K_M 양자화


In [ ]:
!python llama.cpp/convert_hf_to_gguf.py merged7b --outfile v12_7b.f16.gguf --outtype f16
!./llama.cpp/build/bin/llama-quantize v12_7b.f16.gguf qwen-security-v12-7b.Q4_K_M.gguf Q4_K_M
!rm -f v12_7b.f16.gguf
import os; print('Q4 크기(GB):', round(os.path.getsize('qwen-security-v12-7b.Q4_K_M.gguf')/1e9,2))

## ⑤ Q4 GGUF 다운로드 (~4.5GB)
받은 파일을 로컬 `scanops-model/models/`에 두고:
```bash
ollama create qwen2.5-coder-security-v12-7b -f models/Modelfile_v12_7b
# Modelfile의 FROM 경로를 이 GGUF로 맞출 것
```


In [ ]:
from google.colab import files
files.download('qwen-security-v12-7b.Q4_K_M.gguf')